## MODEL 1: Embedding Model

In [4]:
!pip install -U trl bitsandbytes langchain langchain-community faiss-cpu sentence-transformers langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [11]:
pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
#%load_ext autoreload
#%autoreload 2

from atakapa_rag_pipeline import GrammarRAGPipeline
from AtakapaSLMTrainer import AtakapaSLMTrainer
from AtakapaTokenizer import AtakapaTokenizer
from datasets import load_dataset, DatasetDict
import bitsandbytes

In [ ]:
import os

if os.path.exists('atakapa_rag_pipeline.py'):
    print('atakapa_rag_pipeline.py found!')
else:
    print('atakapa_rag_pipeline.py not found. Please ensure it has been uploaded.')

atakapa_rag_pipeline.py not found. Please ensure it has been uploaded.


In [ ]:
GRAMMAR_FILE = "Kimball_Grammar_Layout_Preserved.txt"

rag_pipeline = GrammarRAGPipeline()
rag_pipeline.ingest_grammar_book(GRAMMAR_FILE)


Loading embedding model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading grammar text from Kimball_Grammar_Layout_Preserved.txt
Split Grammar into 581 chunks
Embedding chunks and building FAISS index...
Vector database built successfully!


In [ ]:
rag_pipeline.query_grammar("How does the directional case suffix -ot work?")


--- Searching for: 'How does the directional case suffix -ot work?' ---

[Result 1]
[Yuk’hiˊti ipcōˊk oˊk yaˊ peˊneat]
322) yukhíti ipšók ókya péniat
yukhiti   ipšok-ø   ok=ya     ø-peni-ø-at
Indian    doctor-ABS come=and 3OBJ-cure-3SS-PRET
The Indian doctor came and cured her. (1932:55-56)
Directional case. The directional case is marked by the suffix -ot. Frequently it is written -ut,
the latter a weakened form occurring after an accented syllable. Most examples in the texts and
dictionary are written separately from the noun. This is based on Gatschet’s field notes, for after
he decided that the case marker was a postposition, he often went back and separated it from the
noun it was written together with, and added an accent. However, -ot is not a postposition, as
unlike true postpositions it never occurs verbalized. The basic meaning of the directional case is
action happening to or towards the noun to which it is attached. In terms of grammar, it has both
dative and locative func

In [13]:
dataset = load_dataset("json", data_files="atakapa_training_dataset.jsonl")

full_dataset = dataset["train"]
train_holdout = full_dataset.train_test_split(test_size=0.2, seed=42)
val_test = train_holdout['test'].train_test_split(test_size=0.5, seed=42)

final_dataset = DatasetDict({
    'train': train_holdout['train'],     # ~2,271 examples (80%)
    'validation': val_test['train'],     # ~284 examples (10%)
    'test': val_test['test']             # ~284 examples (10%)
})

In [15]:
# Assuming the AtakapaSLMTrainer class is loaded
trainer = AtakapaSLMTrainer(model_id="Qwen/Qwen2.5-0.5B")

# Get the tokenizer lists from your AtakapaTokenizer class
atakapa_tok = AtakapaTokenizer()
custom_morphemes = list(atakapa_tok.valid_roots) + atakapa_tok.prefix_list + atakapa_tok.suffix_list

# Initialize the model architecture
trainer.setup_model_and_tokenizer(custom_morphemes)

# Train!
trainer.train(final_dataset['train'])

Loading Qwen/Qwen2.5-0.5B...
Added 1864 new tokens (Morphemes + Glosses).


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 283,648,000 || all params: 778,971,008 || trainable%: 36.4132


Adding EOS to train dataset:   0%|          | 0/2271 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss
10,8.368399
20,4.539824
30,2.493892
40,1.848241
50,1.827459
60,1.532902
70,1.494002
80,1.337255
90,1.563826
100,1.347575


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Training complete and model saved.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [17]:
# import shutil
# from google.colab import files

# # Define the name of the zip file
# zip_filename = 'content_backup'

# # Compress the /content directory (excluding the 'drive' folder to avoid recursive errors or huge downloads)
# # We zip everything except the 'drive' mount point.
# !zip -r {zip_filename}.zip /content -x "/content/drive/*"

# # Download the file to your local machine
# files.download(f'{zip_filename}.zip')

Scanning files .......
	zip warning: name not matched: /content/drive/MyDrive/Moonlight.pdf
.	zip warning: name not matched: /content/drive/.Encrypted/.shortcut-targets-by-id/0B68EMBjUKo1mejlPNFlSSEc3MXM/Sarrihn
	zip warning: name not matched: /content/drive/.Encrypted/.shortcut-targets-by-id/0B68EMBjUKo1mSXZpbDNUd3N5ems/Literature
	zip warning: name not matched: /content/drive/.Encrypted/.shortcut-targets-by-id/0B68EMBjUKo1mdk9GUnBaVDd0M2s/Hradrimar
	zip warning: name not matched: /content/drive/.Encrypted/.shortcut-targets-by-id/0B68EMBjUKo1maFVybVd0bVdyeXM/Languages
  adding: content/ (stored 0%)
  adding: content/.config/ (stored 0%)
  adding: content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db (deflated 97%)
  adding: content/.config/.last_opt_in_prompt.yaml (stored 0%)
  adding: content/.config/.last_update_check.json (deflated 22%)
  adding: content/.config/active_config (stored 0%)
  adding: content/.config/configurations/ (stored 0%)
  adding: conten

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# import os

# # Define the target path in Google Drive
# drive_path = '/content/drive/MyDrive/capstone'

# # Create the folder if it doesn't exist
# if not os.path.exists(drive_path):
#     os.makedirs(drive_path)
#     print(f"Created directory: {drive_path}")

# # Copy the zip file to the drive folder
# !cp content_backup.zip "{drive_path}/"

# print(f"Success! 'content_backup.zip' has been saved to your Google Drive in the 'capstone' folder.")

Created directory: /content/drive/MyDrive/capstone
Success! 'content_backup.zip' has been saved to your Google Drive in the 'capstone' folder.


In [19]:
# from google.colab import drive

# # Force a sync between Colab's local mount and the actual Google Drive cloud storage
# drive.flush_and_unmount()

# # Re-mount to confirm the file is visible in the system
# drive.mount('/content/drive')

# # Double check the file exists in the cloud path
# import os
# if os.path.exists('/content/drive/MyDrive/capstone/content_backup.zip'):
#     print("✅ File successfully synced and verified in Drive folder!")
# else:
#     print("❌ File still syncing. Please wait a minute and refresh your Drive browser tab.")

Mounted at /content/drive
✅ File successfully synced and verified in Drive folder!


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 1. Point this to the exact name of your downloaded folder
local_model_path = "./atakapa-glosser-final"
base_model_id = "Qwen/Qwen2.5-0.5B"

print("Loading custom tokenizer...")
# 2. Load YOUR custom tokenizer from the downloaded folder so it recognizes the new morphemes
tokenizer = AutoTokenizer.from_pretrained(local_model_path)

print("Loading base Qwen model...")
# 3. Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto", # Automatically uses GPU if you have one, or CPU if you don't
    dtype=torch.float32 # Use float32 if running on standard CPU memory
)

# Resize the base model's token embeddings to match your custom tokenizer
base_model.resize_token_embeddings(len(tokenizer))

print("Applying fine-tuned Atakapa weights...")
# 4. Fuse your downloaded adapter weights onto the base model
model = PeftModel.from_pretrained(base_model, local_model_path)

print("Model successfully loaded and ready for inference!")

Loading custom tokenizer...


Loading base Qwen model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Applying fine-tuned Atakapa weights...
Model successfully loaded and ready for inference!


In [14]:
# Create a quick test prompt using your exact chat template format
test_sentence = 'hat-wánc'
prompt = f"Gloss the following Atakapa sentence: {test_sentence}"

# Tokenize and generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs, 
    max_new_tokens=50, 
    temperature=0.1, # Keep it low so it doesn't hallucinate weird glosses
    pad_token_id=tokenizer.pad_token_id
)

# Decode the response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Gloss the following Atakapa sentence: hat-wánc
Gloss the following A


In [24]:
# 1. Set up your prompt as a conversational message
test_sentence = 'iyáŋ-ik mò·n n̥-ká-ne'


messages = [
    {"role": "user", "content": f"Gloss the following Atakapa sentence: {test_sentence}"}
]

# 2. Let the tokenizer apply the correct chat template
# add_generation_prompt=True is CRITICAL! It adds the invisible "assistant" trigger token.
formatted_prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

# 3. Tokenize and generate
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs, 
    max_new_tokens=50, 
    temperature=0.1, 
    pad_token_id=tokenizer.pad_token_id,
    repetition_penalty=1.1 # Optional: Adds a slight penalty to discourage looping
)

# 4. Decode the response
# We still skip_special_tokens so the <|im_start|> tags don't print to your screen
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Optional: To only print the assistant's response instead of the whole prompt + response
# print(response.split("assistant\n")[-1].strip()) 
print(response)

system
You are a helpful assistant.
user
Gloss the following Atakapa sentence: iyáŋ-ik mò·n n̥-ká-ne
assistant
someone-ERG.AGNT 2SOBJ-wrap-3PLS-PRET
